In [ ]:
import deepinv as dinv
import torch
import matplotlib.pyplot as plt
import numpy as np
from priors import WeightedTikhonovPrior
from prior_comparison import DegradedLikelihood
from utils import device
from sampling import GaussianDiag, SKROCK


In [ ]:
sigmax = 1. 
sigma = 0.05

def generate_measurements(d, sigmax, sigma):
    d = int(d)
    x = torch.tensor(sigmax*np.random.normal(size=d)).to(device).reshape((1, 1, d))
    
    p = dinv.physics.LinearPhysics(  # identity, for compatibility
        img_size=(1, d),
        device=device)
    
    y = p(x) + torch.tensor(sigma*np.random.normal(size=d)).to(device)
    return y, x, p

def compute_evidence(d, sigmax, sigma, y, mlog=True):  # sum of X and a gaussian of variance sigma^2, -log () if mlog is True
    res = 0.5*np.sum(y**2)/(sigmax**2 + sigma**2)
    res += 0.5*d*np.log(2*np.pi)  # normalization of likelihood term

    if mlog == False:
        res = np.exp(-res)
    return res


def compute_evidence2(d, sigmax, sigma, alpha, y, yp, ym, mlog=True):  # p(y+/y-) 
    sigmax2, sigma2 = sigmax**2, sigma**2 
    res = -0.5*np.sum(y**2)*sigmax2/sigma2/(sigmax2 + sigma2) + 0.5 * alpha**2 * sigmax2 * np.sum(ym**2) / sigma2 / (alpha*sigmax2 + sigma2)
    res += 0.5 * (1-alpha) * np.sum(yp**2) / sigma2
    res += d*np.log(sigma) + 0.5*d*np.log(sigma2 + sigmax2)
    res += 0.5*d*np.log(2*np.pi)
    
    res -= 0.5*d*np.log(1-alpha)
    res -= 0.5*d*np.log(alpha*sigmax2 + sigma2)

    if mlog == False:
        return np.exp(-res)
    else:
        return res
        


### Loop over d

In [ ]:
torch.manual_seed(0)
np.random.seed(0)
alpha = 0.5

nval, ntry = 1, 50  # test nval dimensions, with ntry noise samples
dims = torch.linspace(5, 5., nval, dtype=torch.int)
hist, evidences = np.zeros([nval, ntry]), np.zeros(nval)
lik_traces_exact = [[] for _ in range(nval)]  # list of lists (nval, ntry)
lik_traces = [[] for _ in range(nval)]  # list of lists (nval, ntry)    
hist_exact = np.zeros([nval, ntry])  # using exact p(x/y-)
nb_steps, burnin_ratio = 15000, 0.5
nb_steps_sampler, thinning = int(nb_steps/(1-burnin_ratio)), 5
evidences2 = np.zeros([nval, ntry])  # using p(y+/y-)

# generate measurements
ys, xs, ps = [], [], []
for i in range(nval):
    y, x, p = generate_measurements(dims[i], sigmax, sigma)
    ys.append(y.float()), xs.append(x.float()), ps.append(p)

# generate noise samples 
noises = [torch.randn([1, 1, dims[i], ntry], device=device)*sigma for i in range(nval)]
for i in range(nval):
    d = dims[i]
    y, x, p = ys[i], xs[i], ps[i]
    g = WeightedTikhonovPrior(torch.tensor(1.), torch.eye(d)/sigmax**2)
    # Lipschitz constant of nabla f
    L_f = 1 / (sigma**2 / (1-alpha) )  # divide by 2 because std is sqrt(2)sigma    
    L_g = d / sigmax**2
    gamma = 0.98*1/(L_f + L_g)
    evidences[i] = compute_evidence(d, sigmax, sigma, y.cpu().numpy(), mlog=True)
    print("d={}".format(d))
    for l in range(ntry):
        print("try {}".format(l))
        noise = noises[i][:, :, :, l]
         
        # use a sampler 
        dl = DegradedLikelihood(y, g, p, sigma, gamma, X_init=p.A_A_adjoint(y).to(device).clone(), noise=noise,
                                sampler=SKROCK,sampler_kwargs={'s':15}, lam_reg=None, project=None, alpha=alpha)

        X_post_trace, post_hist, lik_trace, lik_mean = dl.compute_test(nb_steps_sampler, burnin_ratio=burnin_ratio, log_stats=True, thinning=thinning, tol=1e-98, patience=50, normalize=True)
        lik_traces[i].append(lik_trace)
        hist[i, l] = -lik_mean

        evidences2[i, l] = compute_evidence2(d, sigmax, sigma, alpha, y.cpu().numpy(), yp=dl.y_add.cpu().numpy(), ym=dl.y_sub.cpu().numpy(), mlog=True)

        dl = DegradedLikelihood(y, g, p, sigma, gamma, X_init=p.A_A_adjoint(y).to(device).clone(),sampler=GaussianDiag, noise=noise,
                                sampler_kwargs={'d':torch.tensor(d).to(device), 'sigma':torch.tensor(sigma*sigmax / np.sqrt(sigma**2+alpha*sigmax**2)).to(device)},
                                lam_reg=None, project=None, alpha=alpha)
        X_post_trace, post_hist, lik_trace, lik_mean = dl.compute_test(nb_steps, burnin_ratio=0, log_stats=True, thinning=1, tol=1e-98, patience=50, normalize=True)
        lik_traces_exact[i].append(lik_trace)
        hist_exact[i, l] = -lik_mean

        print("p(y+/y-) sampler: {}\n analytical post: {}\n exact: {}".format(hist[i, l], hist_exact[i, l], evidences2[i, l]))

In [ ]:
errors = np.abs(hist - evidences2)
errors_exact = np.abs(hist_exact - evidences2)
mean_errors_exact = np.mean(np.abs(hist_exact - evidences2), axis=1)  
mean_errors_sampler = np.mean(np.abs(hist - evidences2), axis=1) 
print(mean_errors_exact, mean_errors_sampler)
plt.hist([errors.flatten(),errors_exact.flatten() ], bins=20, edgecolor='k', label=['SKROCK', 'Analytical post'])
plt.ylabel(r'$|\log p(y^+/y^-) -\log \hat p(y^+/y^-)|$');
plt.title('Errors, d=5, 15000 iterations, 50 trials, mean errors: \n SKROCK: {:.3f}, Analytical post: {:.3f}'.format(np.mean(errors), np.mean(errors_exact)));
plt.legend();
plt.tight_layout();
plt.savefig('figs/error_hist_15k.pdf', dpi=300)

In [ ]:
# convergence plot 
res, res2 = np.zeros([ntry, nb_steps]), np.zeros([ntry, nb_steps])
for i in range(ntry):
    res[i] = lik_traces[0][i]
    res2[i] = lik_traces_exact[0][i]

ind = 0
val = evidences2[0, ind]

cumsum = -torch.logcumsumexp(torch.tensor(res), 1) + torch.log(torch.arange(1, nb_steps+1))
cumsum2 = -torch.logcumsumexp(torch.tensor(res2), 1) + torch.log(torch.arange(1, nb_steps+1))
err_trace = np.abs(cumsum - evidences2[0, :, None])
err_trace2 = np.abs(cumsum2 - evidences2[0, :, None])
#plt.plot(cumsum[ind], label='SKROCK', color='blue')
#plt.hlines(val, 0, nb_steps, color='blue', linestyle='dashed', label='Analytical post')
plt.plot(np.mean(err_trace.numpy(), axis=0), label='SKROCK', color='blue')
plt.plot(np.mean(err_trace2.numpy(), axis=0), label='Analytical post', color='orange')

plt.ylabel(r'$|\log p(y^+/y^-) -\log \hat p(y^+/y^-)|$');
plt.yscale('log')
plt.xlabel('Iterations')
plt.title('Mean convergence over trials, d=5, 10000 iterations, 50 trials')
plt.legend()
plt.savefig('figs/convergence_15k.pdf', dpi=300)

In [ ]:
mean_errors_exact = np.mean(np.abs(hist_exact - evidences2), axis=1)  
mean_errors_sampler = np.mean(np.abs(hist - evidences2), axis=1) 
ind = 0
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(dims, evidences2, '--', label=r'Analytical $-\log p(y^{+}/y^{-})$')

ax.plot(dims, hist[:, 0], label=r'MC with SKROCK')
ax.plot(dims, hist_exact[:, 0], label=r'MC with analytical $-\log p(x/y^{-})$')
ax.set_xlabel(r'$d$')
ax.legend()